# 03 Create Stratified Pilot Subset 1260

This notebook creates a representative stratified `pilot_1260` subset.

The subset preserves:

- the original dataset proportions between HAM10000 and ISIC2018
- the HAM10000 melanoma / non-melanoma label distribution
- the ISIC2018 mask-size and border-touch diagnostic distribution

Output:

```text
data/pilot_1260/pilot_subset_1260.csv
```


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Paths and settings


BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"

HAM_CSV = DATA_DIR / "preprocessed_manifests" / "ham10000_preprocessed.csv"
ISIC_CSV = DATA_DIR / "preprocessed_manifests" / "isic2018_preprocessed.csv"

RUN_NAME = "pilot_1260_strat"
PILOT_DIR = DATA_DIR /RUN_NAME
PILOT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_CSV = PILOT_DIR / f"{RUN_NAME}.csv"

RANDOM_STATE = 42
TOTAL_TARGET = 1260
ID_COL = "stem"

print(f"BASE_DIR:   {BASE_DIR}")
print(f"DATA_DIR:   {DATA_DIR}")
print(f"RUN_NAME:   {RUN_NAME}")
print(f"PILOT_DIR:  {PILOT_DIR}")
print(f"PILOT_CSV:  {PILOT_CSV}")

In [ ]:
# ============================================================
# Load manifests


ham_df = pd.read_csv(HAM_CSV)
isic_df = pd.read_csv(ISIC_CSV)

ham_df["dataset"] = "HAM10000"
isic_df["dataset"] = "ISIC2018"

assert ID_COL in ham_df.columns, f"Missing {ID_COL} in HAM manifest"
assert ID_COL in isic_df.columns, f"Missing {ID_COL} in ISIC manifest"

print("HAM records:", len(ham_df))
print("ISIC records:", len(isic_df))
print("Total records:", len(ham_df) + len(isic_df))

print("\nHAM columns:")
print(ham_df.columns.tolist())

print("\nISIC columns:")
print(isic_df.columns.tolist())

In [ ]:
# ============================================================
# Helper functions


def allocate_proportional_counts(counts: pd.Series, target_n: int) -> pd.Series:
    """
    Allocate target_n samples proportionally to observed stratum counts.

    Uses largest-remainder rounding so that the final total is exactly target_n.
    """
    counts = counts.astype(int)
    raw = counts / counts.sum() * target_n

    allocation = np.floor(raw).astype(int)
    remainder = target_n - allocation.sum()

    if remainder > 0:
        fractional = (raw - allocation).sort_values(ascending=False)
        for idx in fractional.index[:remainder]:
            allocation.loc[idx] += 1

    elif remainder < 0:
        fractional = (raw - allocation).sort_values(ascending=True)
        removable = fractional[allocation.loc[fractional.index] > 0]
        for idx in removable.index[:abs(remainder)]:
            allocation.loc[idx] -= 1

    assert allocation.sum() == target_n, (
        f"Allocation error: expected {target_n}, got {allocation.sum()}"
    )

    return allocation


def proportional_stratified_sample(
    df: pd.DataFrame,
    target_n: int,
    strata_cols,
    id_col: str = ID_COL,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Sample target_n rows proportionally across one or more strata columns.
    """
    df = df.copy()

    if isinstance(strata_cols, str):
        strata_cols = [strata_cols]

    assert id_col in df.columns, f"Missing ID column: {id_col}"

    for col in strata_cols:
        assert col in df.columns, f"Missing stratification column: {col}"

    df["_stratum"] = (
        df[strata_cols]
        .astype(str)
        .agg("__".join, axis=1)
    )

    counts = df["_stratum"].value_counts()
    allocation = allocate_proportional_counts(counts, target_n)

    parts = []

    for stratum, n in allocation.items():
        if n == 0:
            continue

        group = df[df["_stratum"] == stratum]

        if n > len(group):
            raise ValueError(
                f"Requested {n} samples from stratum {stratum}, "
                f"but only {len(group)} available."
            )

        parts.append(
            group.sample(
                n=n,
                random_state=random_state
            )
        )

    sample = (
        pd.concat(parts)
        .drop(columns=["_stratum"])
        .drop_duplicates(subset=[id_col])
        .reset_index(drop=True)
    )

    assert len(sample) == target_n, f"Expected {target_n}, got {len(sample)}"

    return sample

In [ ]:
# ============================================================
# Dataset-level proportional allocation


dataset_counts = pd.Series({
    "HAM10000": len(ham_df),
    "ISIC2018": len(isic_df),
})

dataset_allocation = allocate_proportional_counts(
    dataset_counts,
    TOTAL_TARGET
)

N_HAM = int(dataset_allocation["HAM10000"])
N_ISIC = int(dataset_allocation["ISIC2018"])

print("Dataset allocation:")
display(dataset_allocation)

print(f"HAM target:  {N_HAM}")
print(f"ISIC target: {N_ISIC}")
print(f"Total:       {N_HAM + N_ISIC}")

In [ ]:
# ============================================================
# HAM10000: representative sampling by class label


assert "label" in ham_df.columns, "HAM manifest must contain a label column"

ham_sample = proportional_stratified_sample(
    ham_df,
    target_n=N_HAM,
    strata_cols=["label"],
    id_col=ID_COL,
    random_state=RANDOM_STATE,
)

print("HAM sample:", len(ham_sample))

print("\nHAM label counts:")
display(ham_sample["label"].value_counts().sort_index())

print("\nHAM label percentages:")
display((ham_sample["label"].value_counts(normalize=True).sort_index() * 100).round(2))

In [ ]:
# ============================================================
# ISIC2018: representative sampling by mask diagnostics


required_isic_cols = ["mask_size_class", "touches_any_border"]
for col in required_isic_cols:
    assert col in isic_df.columns, f"ISIC manifest missing required column: {col}"

isic_sample = proportional_stratified_sample(
    isic_df,
    target_n=N_ISIC,
    strata_cols=["mask_size_class", "touches_any_border"],
    id_col=ID_COL,
    random_state=RANDOM_STATE,
)

print("ISIC sample:", len(isic_sample))

print("\nISIC mask size counts:")
display(isic_sample["mask_size_class"].value_counts())

print("\nISIC mask size percentages:")
display((isic_sample["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nISIC border-touch counts:")
display(isic_sample["touches_any_border"].value_counts())

print("\nISIC border-touch percentages:")
display((isic_sample["touches_any_border"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# ============================================================
# Merge and save pilot subset

pilot_df = (
    pd.concat([ham_sample, isic_sample])
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

assert len(pilot_df) == TOTAL_TARGET, f"Expected {TOTAL_TARGET}, got {len(pilot_df)}"
assert pilot_df[ID_COL].notna().all(), f"Missing values in ID column: {ID_COL}"

pilot_df.to_csv(PILOT_CSV, index=False)

print(f"Saved: {PILOT_CSV}")
print("Total pilot records:", len(pilot_df))

print("\nDataset distribution:")
display(pilot_df["dataset"].value_counts())
display((pilot_df["dataset"].value_counts(normalize=True) * 100).round(2))

print("\nHAM label distribution:")
ham_pilot = pilot_df[pilot_df["dataset"] == "HAM10000"]
display(ham_pilot["label"].value_counts().sort_index())
display((ham_pilot["label"].value_counts(normalize=True).sort_index() * 100).round(2))

display(ham_pilot["mask_size_class"].value_counts())
display((ham_pilot["mask_size_class"].value_counts(normalize=True) * 100).round(2))
print("HAM border-touch distribution:")
display(ham_pilot["touches_any_border"].value_counts())
display((ham_pilot["touches_any_border"].value_counts(normalize=True) * 100).round(2))




print("\nISIC mask size distribution:")
isic_pilot = pilot_df[pilot_df["dataset"] == "ISIC2018"]
display(isic_pilot["mask_size_class"].value_counts())
display((isic_pilot["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nISIC border-touch distribution:")
display(isic_pilot["touches_any_border"].value_counts())
display((isic_pilot["touches_any_border"].value_counts(normalize=True) * 100).round(2))


In [ ]:
# ============================================================
# Optional sanity check: compare full data vs pilot distributions

print("Full dataset allocation:")
display(dataset_counts)
display((dataset_counts / dataset_counts.sum() * 100).round(2))

print("\nPilot dataset allocation:")
display(pilot_df["dataset"].value_counts())
display((pilot_df["dataset"].value_counts(normalize=True) * 100).round(2))

print("\nFull HAM label distribution:")
display(ham_df["label"].value_counts().sort_index())
display((ham_df["label"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nPilot HAM label distribution:")
display(ham_pilot["label"].value_counts().sort_index())
display((ham_pilot["label"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nFull ISIC mask size distribution:")
display(isic_df["mask_size_class"].value_counts())
display((isic_df["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nPilot ISIC mask size distribution:")
display(isic_pilot["mask_size_class"].value_counts())
display((isic_pilot["mask_size_class"].value_counts(normalize=True) * 100).round(2))

print("\nFull ISIC border-touch distribution:")
display(isic_df["touches_any_border"].value_counts())
display((isic_df["touches_any_border"].value_counts(normalize=True) * 100).round(2))

print("\nPilot ISIC border-touch distribution:")
display(isic_pilot["touches_any_border"].value_counts())
display((isic_pilot["touches_any_border"].value_counts(normalize=True) * 100).round(2))